# MIDA507 -- Session 03
## Open Data, Licences Ouvertes et Portails Africains

| | |
|---|---|
| **Programme** | Master MIDA -- Universite de Douala |
| **Session** | S03 -- Open Data, licences et portails africains |
| **Duree estimee** | 90 minutes |
| **Prerequis** | Notebooks S01 et S02 |

### Ce que vous allez apprendre
1. Ce qu'est l'open data et pourquoi il concerne directement l'IDA
2. Les 10 principes Sunlight : l'etalon international de l'open data
3. Comment choisir la bonne licence ouverte pour votre institution
4. Auditer un portail africain avec une grille professionnelle
5. Les 8 barrieres specifiques a l'open data en Afrique et les solutions IDA

### Livrable : `MIDA507_S03_audit_portail.json`

> Cellule grise = code a executer (Shift + Entree). Cellule blanche = texte a lire.


In [ ]:
# INSTALLATION ET CHARGEMENT DES OUTILS
!pip install pandas seaborn matplotlib openpyxl --quiet
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import json, os, hashlib, random
from datetime import datetime, timedelta
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style="whitegrid")
print("Outils installes et charges.")


In [ ]:
random.seed(42)
NB = 5000
TYPES = ["These et memoire","Manuel universitaire","Revue scientifique",
         "Rapport de recherche","Ouvrage de reference","Document administratif"]
FILIERES = ["Droit","Sciences eco.","Lettres","Histoire","Geographie","Medecine","Agronomie","Informatique"]
NIVEAUX = ["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
REGIONS = ["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
LANGUES = ["Francais","Anglais","Bilingue","Arabe","Autres"]
d0 = datetime(2018,1,1)
emprunts = pd.DataFrame({
    "cote":    [f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_doc":random.choices(TYPES, weights=[0.28,0.30,0.15,0.10,0.10,0.07], k=NB),
    "date":    [(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(NB)],
    "duree":   random.choices([3,7,7,14,14,14,21,30], k=NB),
    "usager":  [f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere": random.choices(FILIERES, k=NB),
    "niveau":  random.choices(NIVEAUX, k=NB),
    "region":  random.choices(REGIONS, k=NB),
    "langue":  random.choices(LANGUES, weights=[0.62,0.22,0.10,0.04,0.02], k=NB),
})
emprunts["annee"] = pd.to_datetime(emprunts["date"]).dt.year
print(f"Jeu BU-UNG : {len(emprunts):,} emprunts, {len(emprunts.columns)} colonnes.")


---
## NOTION 1 -- Qu'est-ce que l'Open Data ?

### Definition en 3 lignes

L'**open data** (donnees ouvertes) designe des donnees publiees librement sur Internet, accessibles a tous, gratuitement, sans restriction d'usage, dans un format exploitable par une machine. Ce n'est pas seulement "mettre un fichier en ligne" -- c'est respecter un ensemble de conditions precises.

**Analogie IDA :** L'open data, c'est passer d'un fonds documentaire en acces restreint (salle de lecture sur rendez-vous, avec carte professionnelle) a un fonds entierement numerise, telechargeable, sans inscription, sous licence de reutilisation libre. La **bibliotheque publique numerique universelle**.

### 5 conditions minimales de l'open data

| Condition | Notre jeu BU-UNG | Action requise |
|---|---|---|
| Accessible en ligne via URL stable | Non encore publie | Session S06 (CKAN) |
| Gratuit, sans inscription | Non encore publie | Session S06 |
| Format CSV ou JSON (lisible par machine) | OUI -- deja en CSV | Deja conforme |
| Licence de reutilisation explicite | Aucune licence declaree | Cette session S03 |
| Metadonnees completes (titre, auteur...) | Pas encore de fiche DCAT | Session S04 |

> L'IDA n'est pas un executant technique -- il est le garant de la qualite, de la legalite et de la perennite des donnees ouvertes.


In [ ]:
# NOTION 1 EN PRATIQUE -- Verifions les 5 conditions open data de notre jeu
print("VERIFICATION OPEN DATA -- Jeu BU-UNG (etat actuel)")
print("=" * 60)
print()
conditions = [
    ("1. Accessible en ligne (HTTPS + URL stable)",  False, "Non publie -- a corriger en S06"),
    ("2. Gratuit et sans inscription",               False, "Non publie -- a corriger en S06"),
    ("3. Format CSV lisible par machine",             True,  "Deja conforme -- rien a faire"),
    ("4. Licence de reutilisation declaree",         False, "Choisir CC-BY 4.0 dans cette session"),
    ("5. Metadonnees completes (fiche DCAT)",        False, "A rediger en session S04"),
]
nb_ok = 0
for condition, statut, action in conditions:
    icone = "OK" if statut else "MANQUANT"
    print(f"  [{icone}] {condition}")
    print(f"          --> {action}")
    print()
    if statut: nb_ok += 1
print(f"Resultat : {nb_ok}/5 conditions open data remplies actuellement.")
print("Objectif apres S03-S06 : 5/5")


---
## NOTION 2 -- Les 10 Principes Sunlight

### Definition en 3 lignes

Les **10 principes Sunlight** (2007) sont le referentiel international qui definit ce qu'est l'open data gouvernemental. Ils sont utilises par les Nations Unies, l'Union Africaine et les gouvernements pour evaluer leurs portails.

**Analogie IDA :** C'est l'equivalent des normes de catalogage (ISBD, RDA) pour les bibliotheques -- un referentiel partage internationalement permettant de comparer et evaluer les pratiques, ici applique aux donnees plutot qu'aux documents.

**Chaque principe vaut 10 points -- le score maximal est 100.**


In [ ]:
# NOTION 2 EN PRATIQUE -- Les 10 Principes Sunlight avec scores BU-UNG
principes = [
    ("1. Completude",        "Toutes les donnees collectees sont publiees sans omission.",            8),
    ("2. Primaute",          "Donnees publiees proches de leur source originale, sans agregation.",   8),
    ("3. Actualite",         "Publiees des que possible, frequence de MAJ declaree et respectee.",    5),
    ("4. Accessibilite",     "Accessibles a tous sans inscription ni paiement.",                      0),
    ("5. Lisibilite machine","Format CSV/JSON disponible -- pas seulement PDF ou images.",            9),
    ("6. Non-discrimination","Accessibles a tous sans discrimination de personne ou organisation.",   8),
    ("7. Non-propriete",     "Formats ouverts non controles par une entreprise (CSV, ODS).",         9),
    ("8. Licence libre",     "Licence autorisant la reutilisation, modification, redistribution.",    0),
    ("9. Permanence",        "URLs stables, gestion des versions, archivage des editions.",           2),
    ("10. Gratuite totale",  "Entierement gratuit -- aucun tarif meme pour usage commercial.",       10),
]
print("LES 10 PRINCIPES SUNLIGHT -- Evaluation BU-UNG")
print("=" * 65)
print()
score_total = 0
for nom, definition, score_bung in principes:
    score_total += score_bung
    barre = chr(9608)*score_bung + chr(9617)*(10-score_bung)
    statut = "OK" if score_bung >= 7 else ("~" if score_bung >= 4 else "ABSENT")
    print(f"  [{statut:6}] {nom:<25} : {score_bung}/10  {barre}")
    print(f"            Definition : {definition}")
    if score_bung == 0:
        print(f"            --> ACTION REQUISE dans cette session ou S04-S06")
    print()
print(f"Score Sunlight actuel BU-UNG : {score_total}/100")
print(f"Cible apres sessions S03-S06  : ~87/100")
print()
print("Les 2 scores de 0 a corriger en priorite :")
print("  - Principe 8 (Licence libre)  --> choisir CC-BY 4.0 dans cette session")
print("  - Principe 4 (Accessibilite)  --> publier sur CKAN en session S06")


---
## NOTION 3 -- Les Licences Ouvertes

### Definition en 3 lignes

Une **licence** est un contrat juridique qui indique ce que les reutilisateurs ont le droit de faire avec vos donnees. **Sans licence, toute reutilisation est juridiquement interdite** -- meme si le fichier est accessible en ligne. C'est la lacune la plus frequente des portails africains.

**Analogie IDA :** La licence, c'est la mention de copyright et de droits de reproduction sur un document, la note qui indique "Reproduction autorisee avec mention de source". Sans elle, l'usager ne sait pas ce qu'il peut faire.

### Recommandation IDA pour la BU-UNG : CC-BY 4.0

| Licence | Ce qu'elle permet | Obligation | Recommandee ? |
|---|---|---|---|
| CC0 | Tout, sans restriction | Aucune | Pour donnees meteorologiques, geographiques |
| **CC-BY 4.0** | **Tout** | **Citer la source** | **OUI -- institutions publiques africaines** |
| CC-BY-SA | Tout | Citer + meme licence | Deconseille (contrainte sur les reutilisateurs) |
| CC-BY-NC | Non commercial seulement | Citer | Deconseille pour open data gouvernemental |
| Etalab v2 | Tout | Mentionner source + date | Oui -- portails gouvernementaux francophones |

**Raisons du choix CC-BY pour la BU-UNG :**
1. Institution publique -- les donnees appartiennent au public
2. Maximise la reutilisation par chercheurs, ONG et journalistes
3. Compatible avec les exigences AFD, Carnegie, IDRC, Union Europeenne
4. L'obligation de citation valorise l'institution (pas une contrainte)


In [ ]:
# NOTION 3 EN PRATIQUE -- On declare la licence de notre jeu
print("DECLARATION DE LICENCE -- Jeu BU-UNG")
print("=" * 55)
print()

# Declaration formelle de la licence choisie
licence_choisie = {
    "nom_complet":  "Creative Commons Attribution 4.0 International",
    "abreviation":  "CC-BY 4.0",
    "uri_officielle": "https://creativecommons.org/licenses/by/4.0/",
    "symbole":      "CC BY",
    "obligations_reutilisateur": [
        "Citer : Bibliotheque Centrale -- Universite de Ngaoundere",
        "Indiquer l'URL du jeu de donnees",
        "Indiquer si vous avez modifie les donnees",
    ],
    "ce_qui_est_autorise": [
        "Redistribution du jeu original",
        "Modification et creation de versions derivees",
        "Usage commercial (articles, rapports, produits)",
        "Incorporation dans d'autres bases de donnees",
    ],
    "ce_qui_est_interdit": [
        "Aucune restriction supplementaire -- CC-BY est la licence la plus libre avec attribution",
    ],
}

print(f"  Licence choisie : {licence_choisie['nom_complet']}")
print(f"  Abreviation     : {licence_choisie['abreviation']}")
print(f"  URI officielle  : {licence_choisie['uri_officielle']}")
print()
print("  CE QUE PEUVENT FAIRE LES REUTILISATEURS :")
for autorisation in licence_choisie["ce_qui_est_autorise"]:
    print(f"    OK  {autorisation}")
print()
print("  OBLIGATION UNIQUE :")
for obligation in licence_choisie["obligations_reutilisateur"]:
    print(f"    !   {obligation}")
print()
print("EXEMPLE DE CITATION QUE DEVRONT FAIRE LES REUTILISATEURS :")
annee = datetime.now().year
print(f"  'Bibliotheque Centrale -- Universite de Ngaoundere ({annee}).")
print(f"   Emprunts documentaires BU-UNG 2018-2023. Licence CC-BY 4.0.")
print(f"   URL : https://ckan.demo.ung.cm/dataset/ung-biblio-emprunts-2018-2023'")
print()
print("  --> Cette citation apparaitra dans les articles de recherche,")
print("      les rapports ONG et les publications journalistiques.")
print("      C'est une reconnaissance academique de la BU-UNG.")
# Sauvegarde
with open("MIDA507_S03_licence_bung.json","w",encoding="utf-8") as f:
    json.dump(licence_choisie, f, ensure_ascii=False, indent=2)
print()
print("OK Licence declaree et sauvegardee : MIDA507_S03_licence_bung.json")


---
## NOTION 4 -- Auditer un Portail Open Data Africain

### Definition en 3 lignes

**Auditer un portail** signifie evaluer sa conformite aux principes Sunlight, identifier ses lacunes et recommander des actions d'amelioration. C'est une competence IDA de base pour accompagner les institutions africaines dans leur transition vers l'open data.

**Analogie IDA :** C'est l'equivalent d'un audit de fonds documentaire en bibliotheque -- evaluer selon des criteres professionnels et rediger un rapport de recommandations. Mais pour des portails de donnees au lieu de collections physiques.

### Les 3 niveaux de maturite open data

| Niveau | Score Sunlight | Caracteristiques |
|---|---|---|
| Insuffisant | 0-49 | Pas de licence, format PDF, pas d'API, URL instables |
| Partiel | 50-74 | Licence presente, CSV disponible, mais metadonnees incompletes |
| Conforme | 75-100 | Standards DCAT, API REST, licence CC, preservation assuree |


In [ ]:
# NOTION 4 EN PRATIQUE -- Audit de 3 portails africains
portails = {
    "data.gouv.bj (Benin)": {
        "plateforme": "CKAN (standard open source)",
        "https": True, "inscription": False, "api": True,
        "formats": ["CSV","JSON","XLS"],
        "licence": "Etalab v2.0",
        "scores": [8,8,7,10,9,10,9,9,7,10]
    },
    "opendata.sn (Senegal -- ANSD)": {
        "plateforme": "CKAN",
        "https": True, "inscription": False, "api": True,
        "formats": ["CSV","XLS","PDF"],
        "licence": "Conditions specifiques ANSD",
        "scores": [6,7,6,8,8,9,8,7,6,10]
    },
    "INS-CM hypothetique (Cameroun)": {
        "plateforme": "Portail personnalise",
        "https": False, "inscription": True, "api": False,
        "formats": ["XLS","PDF"],
        "licence": "Aucune",
        "scores": [4,4,3,5,4,7,4,2,3,8]
    }
}

print("AUDIT COMPARATIF -- Portails Open Data Africains")
print("=" * 65)
print()

for nom_portail, info in portails.items():
    score = sum(info["scores"])
    barre = chr(9608)*(score//5) + chr(9617)*(20-score//5)
    niveau = "EXCELLENT" if score>=85 else ("BON" if score>=70 else ("PARTIEL" if score>=50 else "INSUFFISANT"))
    icone = "TOP" if score>=70 else ("MED" if score>=50 else "BAS")

    print(f"  [{icone}] {nom_portail}")
    print(f"       Plateforme  : {info['plateforme']}")
    print(f"       HTTPS       : {'Oui' if info['https'] else 'Non (risque securite)'}")
    print(f"       Inscription : {'Requise (barriere acces)' if info['inscription'] else 'Non requise (conforme)'}")
    print(f"       API REST    : {'Disponible' if info['api'] else 'Absente'}")
    print(f"       Formats     : {', '.join(info['formats'])}")
    print(f"       Licence     : {info['licence']}")
    print(f"       Score       : {score}/100  {barre}  ({niveau})")
    print()

print("LECON CLE :")
print("  data.gouv.bj (Benin) = reference en Afrique francophone avec 87/100.")
print("  Notre BU-UNG vise ce niveau apres les sessions S03 a S06.")
print("  Le Cameroun a encore beaucoup de chemin a parcourir.")


---
## NOTION 5 -- Les 8 Barrieres a l'Open Data en Afrique

> Ces 8 barrieres sont specifiques au contexte africain. L'IDA joue un role cle dans chacune d'elles.


In [ ]:
# NOTION 5 EN PRATIQUE -- 8 barrieres et solutions IDA
barrieres = [
    ("Culturelle",       "Mentalite 'les donnees = actif a proteger', peur de la transparence.",
     "Former les cadres dirigeants, montrer des cas africains de valorisation."),
    ("Technique",        "Serveurs absents, connexions lentes, manque de competences informatiques.",
     "CKAN heberge cloud (10 USD/mois), Zenodo gratuit (5 Go), GitHub Pages."),
    ("Juridique",        "Vide legislatif sur l'open data dans la plupart des pays africains.",
     "Adopter CC-BY ou Etalab meme sans cadre legal national -- la licence fait loi."),
    ("Financiere",       "Aucun budget dedie a la mise en ligne et la maintenance des portails.",
     "CKAN gratuit, Zenodo gratuit, GitHub Pages -- cout quasi nul."),
    ("Qualite",          "Donnees incompletes, formats heterogenes, doublons non publiables.",
     "Pipeline nettoyage Python (session S05) + rapport qualite DAMA avant publication."),
    ("Interoperabilite", "Jeux non connectables entre eux, pas d'identifiants communs.",
     "ARK persistants, standard DCAT, vocabulaires AGROVOC/RAMEAU, codes ISO 639."),
    ("Politique",        "Ministeres opposant la publication de donnees jugees sensibles.",
     "Publier les agreges et anonymises -- confidentialite et open data compatibles."),
    ("Durabilite",       "Portails abandonnes 2 ans apres lancement, URLs mortes, donnees perdues.",
     "ARK + Zenodo pour preservation, politique de maintenance dans le DMP (section 7)."),
]
print("LES 8 BARRIERES A L'OPEN DATA EN AFRIQUE -- Solutions IDA")
print("=" * 65)
print()
for i, (nom, probleme, solution) in enumerate(barrieres, 1):
    print(f"  Barriere {i} -- {nom.upper()}")
    print(f"    Probleme  : {probleme}")
    print(f"    Solution  : {solution}")
    print()
print("OBSERVATION IMPORTANTE :")
print("  6 des 8 barrieres NE SONT PAS des problemes techniques.")
print("  Gouvernance, droit, culture : c'est le domaine de l'IDA, pas de l'informaticien.")


---
## EXERCICE -- Auditez un portail africain de votre choix


In [ ]:
# EXERCICE -- Auditez un portail avec la grille Sunlight
# Remplacez les valeurs entre guillemets et les None.

MON_PORTAIL   = "[A COMPLETER : ex. Portail de l'INS Cameroun]"
MON_PAYS      = "[A COMPLETER : ex. Cameroun]"

# Notez chaque principe de 0 (absent) a 10 (parfaitement conforme)
mes_scores = {
    "1. Completude":         None,
    "2. Primaute":           None,
    "3. Actualite":          None,
    "4. Accessibilite":      None,
    "5. Lisibilite machine": None,
    "6. Non-discrimination": None,
    "7. Non-propriete":      None,
    "8. Licence libre":      None,
    "9. Permanence":         None,
    "10. Gratuite":          None,
}
MON_COMMENTAIRE = "[A COMPLETER : force et lacune principale du portail]"

valides = {k:v for k,v in mes_scores.items() if isinstance(v,(int,float)) and 0<=v<=10}
print(f"AUDIT SUNLIGHT -- {MON_PORTAIL} ({MON_PAYS})")
print("=" * 55)
for principe, note in mes_scores.items():
    if note is None:
        print(f"  [?] {principe:<30} : A evaluer (0 a 10)")
    else:
        barre = chr(9608)*int(note) + chr(9617)*(10-int(note))
        statut = "OK" if note>=7 else ("~" if note>=4 else "X")
        print(f"  [{statut}] {principe:<30} : {note}/10  {barre}")
if valides:
    total = sum(valides.values()); maxi = len(valides)*10
    pct = int(total/maxi*100)
    print(f"\nScore : {total}/{maxi} ({pct}%)")
    print(f"Commentaire : {MON_COMMENTAIRE}")
    audit = {"portail":MON_PORTAIL,"pays":MON_PAYS,"date":datetime.now().strftime("%Y-%m-%d"),
             "scores":mes_scores,"total":total,"commentaire":MON_COMMENTAIRE}
    with open("MIDA507_S03_audit_portail.json","w",encoding="utf-8") as f:
        json.dump(audit,f,ensure_ascii=False,indent=2)
    print("OK Audit sauvegarde : MIDA507_S03_audit_portail.json")
else:
    print("\n[A COMPLETER] Entrez des notes de 0 a 10 pour calculer votre score.")


---
## Bilan de la Session 03

| Competence | Acquise |
|---|---|
| Expliquer l'open data et ses 5 conditions minimales | OK |
| Evaluer les 10 principes Sunlight sur un jeu africain | OK |
| Choisir et declarer la licence CC-BY 4.0 | OK |
| Auditer et comparer des portails africains | OK |
| Identifier les 8 barrieres africaines et solutions IDA | OK |
| Auditer un portail de mon choix | A completer |

### Prochaine session
**S04 -- Metadonnees DCAT :** rediger la notice bibliographique numerique de notre jeu selon le standard international W3C.

*Notebook MIDA507 S03 -- Master MIDA -- Universite de Douala*
